# Mask R-CNN ResNet50-FPN — Segmentación Vertebral
## Con Preprocesamiento Basado en Curva de Columna

| | **Modelo C** | **Modelo D** |
|---|---|---|
| **Arquitectura** | MaskRCNN R50-FPN | MaskRCNN R50-FPN |
| **Dataset** | Filtrado (T1-L1 mín.) | Completo (174 imgs) |
| **Preprocesamiento** | Curva columna + CLAHE | Curva columna + CLAHE |

### Preprocesamiento especial — 3 etapas:
1. **CLAHE** — mejora contraste local de la radiografía (vértebras vs tejido blando)
2. **Curva de columna** (RadiographMetrics) — ajuste de brillo/contraste guiado por la
   curva espinal anotada: píxeles cerca de la curva reciben mayor peso visual
3. **Crop ROI** — recorte centrado en la columna usando la máscara binaria

### Rutas del dataset (tomadas del notebook YOLOv8):
```
Scoliosis_Dataset/
  Normal/           Scoliosis/        ← radiografías
  LabelMultiClass_ID_PNG/             ← máscaras IDs 0-17
  LabelBinaryJPG/                     ← máscara binaria (columna completa)
  RadiographMetrics/                  ← curvas de columna en píxeles
  indice_dataset.csv                  ← índice con rutas
```

## 0 — Instalación

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}')
print(f'GPU:     {torch.cuda.get_device_name(0)}')
print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
import detectron2
print(f'Detectron2: {detectron2.__version__}')

In [ ]:
import os, json, shutil, random, glob
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.model_selection import train_test_split
from scipy.interpolate import UnivariateSpline
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
import detectron2.data as d2_data
import detectron2.data.transforms as T

from google.colab import drive
drive.mount('/content/drive')

# ── RUTAS (mismas que notebook YOLOv8) ───────────────────────
DRIVE_ROOT    = Path('/content/drive/MyDrive')
DATASET_ROOT  = DRIVE_ROOT / 'Scoliosis_Dataset'
CSV_PATH      = DATASET_ROOT / 'indice_dataset.csv'
METRICS_DIR   = DATASET_ROOT / 'RadiographMetrics'   # curvas de columna
BINARY_DIR    = DATASET_ROOT / 'LabelBinaryJPG'       # máscaras binarias
YOLO_DS       = Path('/content/yolo_spine/dataset')          # Dataset A completo
YOLO_DS_FILT  = Path('/content/yolo_spine/dataset_filtered') # Dataset B filtrado

WORK_DIR      = Path('/content/maskrcnn_spine')
COCO_DIR      = WORK_DIR / 'coco_splits'
IMG_PROC_DIR  = WORK_DIR / 'images_preprocessed'  # imágenes preprocesadas
OUTPUT_C      = WORK_DIR / 'output_C_filtrado'
OUTPUT_D      = WORK_DIR / 'output_D_completo'

# ── COLUMNAS CSV ──────────────────────────────────────────────
COL_SPLIT  = 'split'
COL_IMAGE  = 'radiograph_path'
COL_MASK   = 'multiclass_id_png'
COL_BINARY = 'label_binary_path'

# ── CLASES ────────────────────────────────────────────────────
CLASS_NAMES    = ['T1','T2','T3','T4','T5','T6','T7','T8','T9','T10','T11','T12',
                  'L1','L2','L3','L4','L5']
NUM_CLASSES    = 17
MASK_ID_TO_CAT = {i: i for i in range(1, 18)}
SEED           = 42

random.seed(SEED)
np.random.seed(SEED)
for d in [COCO_DIR, OUTPUT_C, OUTPUT_D]:
    d.mkdir(parents=True, exist_ok=True)

print('✔ Configuración lista')
print(f'  METRICS_DIR existe: {METRICS_DIR.exists()}')

---
## 1 — Exploración de RadiographMetrics

Antes de usarlos, exploramos qué formato tienen los archivos de métricas.

In [ ]:
# ── Explorar estructura de RadiographMetrics ──────────────────
if METRICS_DIR.exists():
    files = sorted(METRICS_DIR.iterdir())[:10]
    print(f'Total archivos en RadiographMetrics: {len(list(METRICS_DIR.iterdir()))}')
    print(f'\nPrimeros 10 archivos:')
    for f in files:
        print(f'  {f.name}  ({f.stat().st_size/1e3:.1f} KB)')

    # Leer el primero para ver su estructura
    sample_file = files[0]
    print(f'\nContenido de {sample_file.name}:')
    if sample_file.suffix == '.json':
        with open(sample_file) as f:
            data = json.load(f)
        print(json.dumps(data, indent=2)[:1500])
    elif sample_file.suffix == '.csv':
        print(pd.read_csv(sample_file).head(10).to_string())
    else:
        with open(sample_file) as f:
            print(f.read()[:1000])
else:
    print('⚠ RadiographMetrics no existe o no está montado todavía')

In [ ]:
# ── Función para cargar la curva de columna ───────────────────
# Esta función se adapta automáticamente según el formato detectado
# Ajusta las claves si tu JSON tiene nombres diferentes

def load_spine_curve(image_stem, metrics_dir):
    """
    Carga la curva de la columna para una imagen.
    Retorna array (N, 2) con puntos (x, y) en píxeles, o None si no existe.

    Busca el archivo de métricas con el mismo stem que la imagen.
    Soporta formato JSON y CSV.
    """
    metrics_dir = Path(metrics_dir)

    # Buscar archivo con mismo stem (JSON o CSV)
    for ext in ['.json', '.csv', '.txt']:
        candidate = metrics_dir / f'{image_stem}{ext}'
        if candidate.exists():
            if ext == '.json':
                with open(candidate) as f:
                    data = json.load(f)
                # Intentar varias claves comunes
                for key in ['spine_curve', 'curve', 'points', 'centerline',
                            'vertebrae_centers', 'coords', 'pixels']:
                    if key in data:
                        pts = np.array(data[key])
                        if pts.ndim == 1:
                            pts = pts.reshape(-1, 2)
                        return pts
                # Si no encuentra clave conocida, tomar el primer array
                for v in data.values():
                    if isinstance(v, list) and len(v) > 3:
                        pts = np.array(v)
                        if pts.ndim == 2 and pts.shape[1] == 2:
                            return pts
            elif ext == '.csv':
                df = pd.read_csv(candidate)
                # Buscar columnas x, y
                xcol = next((c for c in df.columns if 'x' in c.lower()), None)
                ycol = next((c for c in df.columns if 'y' in c.lower()), None)
                if xcol and ycol:
                    return df[[xcol, ycol]].values.astype(float)
                elif len(df.columns) >= 2:
                    return df.iloc[:, :2].values.astype(float)
    return None


# ── Verificar con una imagen de ejemplo ──────────────────────
df_check = pd.read_csv(CSV_PATH, sep=';')
test_stem = Path(df_check.iloc[0][COL_IMAGE]).stem
curve = load_spine_curve(test_stem, METRICS_DIR)

if curve is not None:
    print(f'✔ Curva cargada para {test_stem}: {curve.shape} puntos')
    print(f'  Rango X: [{curve[:,0].min():.0f}, {curve[:,0].max():.0f}]')
    print(f'  Rango Y: [{curve[:,1].min():.0f}, {curve[:,1].max():.0f}]')
else:
    print(f'⚠ No se encontró curva para {test_stem}')
    print(f'  Archivos disponibles con ese nombre:')
    for f in METRICS_DIR.glob(f'{test_stem}*'):
        print(f'    {f.name}')

---
## 2 — Preprocesamiento

### Pipeline de preprocesamiento:

```
RX original
    ↓
1. CLAHE (clip=3, grid=8×8)  → mejora contraste local
    ↓
2. Spine Probability Map     → mapa de probabilidad basado en curva
    → para cada Y, la curva define el centro X esperado
    → mapa gaussiano 2D: máximo en la curva, decae con distancia
    ↓
3. Fusión imagen + mapa      → realce suave de la región vertebral
    ↓
4. Crop ROI (máscara binaria) → elimina fondo irrelevante
    ↓
Imagen preprocesada (3 canales para MaskRCNN)
```

In [ ]:
def apply_clahe(gray, clip_limit=3.0, tile_grid=(8, 8)):
    """CLAHE para mejorar contraste local en RX."""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    return clahe.apply(gray)


def build_spine_probability_map(curve_pts, h, w, sigma_x=40):
    """
    Construye mapa de probabilidad 2D basado en la curva de columna.

    Para cada fila Y, la curva define el centro horizontal X esperado.
    Se ajusta un spline a los puntos (Y→X) y se evalúa en cada fila.
    El mapa tiene forma gaussiana: máximo en la curva, decae lateralmente.

    Args:
        curve_pts: array (N, 2) con puntos (x, y) en píxeles
        h, w:      dimensiones de la imagen
        sigma_x:   ancho de la gaussiana en píxeles (≈ ancho de una vértebra)

    Returns:
        mapa float32 normalizado [0, 1], shape (h, w)
    """
    prob_map = np.zeros((h, w), dtype=np.float32)

    # Ordenar puntos por Y
    pts = curve_pts[np.argsort(curve_pts[:, 1])]
    ys  = pts[:, 1].astype(float)
    xs  = pts[:, 0].astype(float)

    # Eliminar duplicados en Y
    _, unique_idx = np.unique(ys, return_index=True)
    ys = ys[unique_idx]
    xs = xs[unique_idx]

    if len(ys) < 4:
        return prob_map

    # Ajustar spline Y → X (la curva puede ser no lineal en escoliosis)
    try:
        spline = UnivariateSpline(ys, xs, k=min(3, len(ys)-1), s=len(ys)*10)
    except Exception:
        # Fallback: interpolación lineal
        spline = lambda y: np.interp(y, ys, xs)

    # Construir mapa fila por fila
    all_y = np.arange(h, dtype=float)
    center_x = np.clip(spline(all_y), 0, w-1)

    x_grid = np.arange(w, dtype=float)
    for row_y in range(h):
        dist   = x_grid - center_x[row_y]
        prob_map[row_y] = np.exp(-0.5 * (dist / sigma_x) ** 2)

    return prob_map


def compute_roi(binary_mask, margin=0.08):
    """Calcula ROI desde máscara binaria con margen."""
    rows = np.any(binary_mask, axis=1)
    cols = np.any(binary_mask, axis=0)
    if not rows.any():
        h, w = binary_mask.shape
        return (0, 0, w, h)
    y1, y2 = np.where(rows)[0][[0, -1]]
    x1, x2 = np.where(cols)[0][[0, -1]]
    h, w   = binary_mask.shape
    dy = max(1, int((y2-y1)*margin))
    dx = max(1, int((x2-x1)*margin))
    return (max(0,x1-dx), max(0,y1-dy), min(w,x2+dx), min(h,y2+dy))


def preprocess_image(img_path, binary_path, curve_pts,
                     target_size=(1024, 1024),
                     use_spine_map=True,
                     spine_map_alpha=0.3):
    """
    Pipeline completo de preprocesamiento.

    Args:
        img_path:      ruta a la radiografía
        binary_path:   ruta a la máscara binaria de columna
        curve_pts:     array (N,2) con curva de columna, o None
        target_size:   (w, h) de salida
        use_spine_map: si True, aplica el mapa de probabilidad
        spine_map_alpha: peso del mapa (0=sin efecto, 1=solo mapa)

    Returns:
        img_out: uint8 (h, w, 3) — imagen preprocesada 3 canales
    """
    # Leer imagen
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    H, W = gray.shape

    # 1. CLAHE
    enhanced = apply_clahe(gray)

    # 2. Mapa de probabilidad espinal
    if use_spine_map and curve_pts is not None and len(curve_pts) >= 4:
        prob_map = build_spine_probability_map(curve_pts, H, W, sigma_x=45)
        # Fusión: realzar región vertebral suavemente
        # enhanced_final = enhanced * (1 + alpha * prob_map)
        enhanced_f = enhanced.astype(np.float32)
        enhanced_f = enhanced_f * (1.0 + spine_map_alpha * prob_map)
        enhanced   = np.clip(enhanced_f, 0, 255).astype(np.uint8)

    # 3. Crop ROI usando máscara binaria
    binary = cv2.imread(str(binary_path), cv2.IMREAD_GRAYSCALE)
    if binary is not None:
        binary_bin = (binary > 127).astype(np.uint8)
        x1, y1, x2, y2 = compute_roi(binary_bin)
        enhanced = enhanced[y1:y2, x1:x2]

    # 4. Resize
    tw, th = target_size
    enhanced = cv2.resize(enhanced, (tw, th), interpolation=cv2.INTER_LINEAR)

    # 5. Convertir a 3 canales (MaskRCNN espera RGB)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)


print('✔ Funciones de preprocesamiento definidas')

In [ ]:
# ── Verificación visual del preprocesamiento ──────────────────
df_check = pd.read_csv(CSV_PATH, sep=';')

# Tomar una imagen de ejemplo
row = df_check.iloc[5]
img_path    = DATASET_ROOT / row[COL_IMAGE]
binary_path = DATASET_ROOT / row[COL_BINARY]
stem        = Path(row[COL_IMAGE]).stem
curve_pts   = load_spine_curve(stem, METRICS_DIR)

# Sin preprocesamiento
orig = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)

# Con preprocesamiento (sin mapa espinal)
proc_no_map = preprocess_image(img_path, binary_path, curve_pts,
                                use_spine_map=False)
proc_no_map = cv2.cvtColor(proc_no_map, cv2.COLOR_BGR2RGB)

# Con preprocesamiento + mapa espinal
proc_with_map = preprocess_image(img_path, binary_path, curve_pts,
                                  use_spine_map=True, spine_map_alpha=0.3)
proc_with_map = cv2.cvtColor(proc_with_map, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 3, figsize=(18, 8))
axes[0].imshow(orig);          axes[0].set_title('Original')
axes[1].imshow(proc_no_map);   axes[1].set_title('CLAHE + Crop ROI')
axes[2].imshow(proc_with_map); axes[2].set_title('CLAHE + Mapa Espinal + Crop ROI')
for ax in axes: ax.axis('off')

if curve_pts is not None:
    print(f'✔ Curva cargada: {len(curve_pts)} puntos')
    # Visualizar el mapa de probabilidad
    gray_orig = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    H, W = gray_orig.shape
    prob_map = build_spine_probability_map(curve_pts, H, W)
    fig2, axes2 = plt.subplots(1, 2, figsize=(12, 7))
    axes2[0].imshow(gray_orig, cmap='gray')
    axes2[0].plot(curve_pts[:,0], curve_pts[:,1], 'r-', lw=2, label='Curva columna')
    axes2[0].legend(); axes2[0].set_title('RX + Curva de columna')
    axes2[1].imshow(prob_map, cmap='hot')
    axes2[1].set_title('Mapa de probabilidad espinal')
    plt.suptitle('RadiographMetrics → Mapa de probabilidad', fontsize=12)
    plt.tight_layout(); plt.show()
else:
    print('⚠ Sin curva para esta imagen — preprocesamiento sin mapa espinal')

plt.suptitle(f'Preprocesamiento — {stem}', fontsize=12)
plt.tight_layout(); plt.show()

---
## 3 — Split y Filtrado

In [ ]:
df = pd.read_csv(CSV_PATH, sep=';')
print(f'Total: {len(df)}')
print(df[COL_SPLIT].value_counts().to_string())

# Split estratificado (mismo SEED que notebook YOLOv8)
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df[COL_SPLIT], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df[COL_SPLIT], random_state=SEED
)
for d in [train_df, val_df, test_df]:
    d.reset_index(drop=True, inplace=True)
print(f'\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

In [ ]:
# ── Obtener stems filtrados (Dataset C) ───────────────────────
# Intenta reutilizar Dataset B del notebook YOLOv8
# Si no existe, filtra desde cero leyendo las máscaras PNG

def stems_from_yolo(yolo_filt, split):
    d = Path(yolo_filt) / 'labels' / split
    return {p.stem for p in d.glob('*.txt')} if d.exists() else set()

def has_min_classes(mask_path, min_classes=13):
    m = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
    if m is None: return False
    if m.ndim == 3: m = m[:,:,0]
    present = set(int(v) for v in np.unique(m) if v > 0)
    return all(i in present for i in range(1, min_classes+1))

# Intentar cargar desde Dataset B existente
train_stems = stems_from_yolo(YOLO_DS_FILT, 'train')

if not train_stems:
    print('Dataset B no encontrado — filtrando desde máscaras PNG (tarda ~3 min)...')
    MIN_CLASSES = 13
    complete_stems = set()
    for _, row in pd.concat([train_df, val_df, test_df]).iterrows():
        mp = DATASET_ROOT / row[COL_MASK]
        if has_min_classes(mp, MIN_CLASSES):
            complete_stems.add(Path(row[COL_IMAGE]).stem)
    train_stems = stems_from_yolo_or_set = complete_stems
    val_stems   = complete_stems
    test_stems  = complete_stems
    print(f'  Imágenes completas encontradas: {len(complete_stems)}')
else:
    val_stems  = stems_from_yolo(YOLO_DS_FILT, 'val')
    test_stems = stems_from_yolo(YOLO_DS_FILT, 'test')
    print(f'Dataset B encontrado en disco')

def filter_df(df, stems):
    return df[df[COL_IMAGE].apply(
        lambda p: Path(p).stem in stems
    )].reset_index(drop=True)

train_f = filter_df(train_df, train_stems)
val_f   = filter_df(val_df,   val_stems)
test_f  = filter_df(test_df,  test_stems)

# Dataset D: completo
train_full = train_df.copy()
val_full   = val_df.copy()
test_full  = test_df.copy()

print(f'\nDataset C (filtrado): Train={len(train_f)} Val={len(val_f)} Test={len(test_f)}')
print(f'Dataset D (completo): Train={len(train_full)} Val={len(val_full)} Test={len(test_full)}')

---
## 4 — Construir Imágenes Preprocesadas y COCO JSONs

Guarda las imágenes preprocesadas en disco y construye los COCO JSONs
apuntando a esas imágenes. Las máscaras GT no se modifican.

In [ ]:
CATEGORIES = [{'id': i, 'name': CLASS_NAMES[i-1]} for i in range(1, NUM_CLASSES+1)]

def mask_to_anns(mask_path, image_id, ann_id, min_area=80):
    """Convierte máscara PNG a anotaciones COCO."""
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
    if mask is None: return [], ann_id
    if mask.ndim == 3: mask = mask[:,:,0]

    # IMPORTANTE: las máscaras GT se escalan al mismo target_size que las imágenes
    H_PROC, W_PROC = 1024, 1024
    if mask.shape != (H_PROC, W_PROC):
        mask = cv2.resize(mask, (W_PROC, H_PROC), interpolation=cv2.INTER_NEAREST)

    h, w = mask.shape
    anns = []
    for mid, cid in MASK_ID_TO_CAT.items():
        binary = (mask == mid).astype(np.uint8) * 255
        if binary.sum() == 0: continue
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = [c for c in cnts if cv2.contourArea(c) >= min_area]
        if not cnts: continue
        cnt  = max(cnts, key=cv2.contourArea)
        area = float(cv2.contourArea(cnt))
        x, y, bw, bh = cv2.boundingRect(cnt)
        seg  = [cnt.reshape(-1,2).astype(float).flatten().tolist()]
        anns.append({'id':ann_id,'image_id':image_id,'category_id':cid,
                     'segmentation':seg,'bbox':[float(x),float(y),float(bw),float(bh)],
                     'area':area,'iscrowd':0})
        ann_id += 1
    return anns, ann_id


def build_dataset(split_df, split_name, suffix, ann_id_start=1,
                  use_spine_map=True):
    """
    Preprocesa imágenes, las guarda en disco y construye el COCO JSON.

    suffix: 'c' o 'd' para diferenciar los splits de cada dataset
    """
    out_img_dir = IMG_PROC_DIR / suffix / split_name
    out_img_dir.mkdir(parents=True, exist_ok=True)

    images, annotations = [], []
    ann_id = ann_id_start
    skipped = 0

    for img_id, (_, row) in enumerate(split_df.iterrows(), 1):
        ip     = DATASET_ROOT / row[COL_IMAGE]
        mp     = DATASET_ROOT / row[COL_MASK]
        bp     = DATASET_ROOT / row[COL_BINARY]
        stem   = ip.stem

        # Cargar curva de columna
        curve  = load_spine_curve(stem, METRICS_DIR)

        # Preprocesar imagen
        proc   = preprocess_image(ip, bp, curve,
                                   target_size=(1024, 1024),
                                   use_spine_map=use_spine_map)
        if proc is None:
            skipped += 1
            continue

        # Guardar imagen preprocesada
        dst_img = out_img_dir / f'{stem}.jpg'
        cv2.imwrite(str(dst_img), proc, [cv2.IMWRITE_JPEG_QUALITY, 95])

        images.append({'id':img_id,'file_name':str(dst_img),
                       'width':1024,'height':1024})

        # Convertir máscara a COCO (escalada a 1024×1024)
        anns, ann_id = mask_to_anns(mp, img_id, ann_id)
        annotations.extend(anns)

    out_json = COCO_DIR / f'{split_name}_{suffix}.json'
    with open(out_json, 'w') as f:
        json.dump({'images':images,'annotations':annotations,
                   'categories':CATEGORIES}, f)

    print(f'  {split_name:6s}_{suffix}: {len(images)} imgs, '
          f'{len(annotations)} anns, {skipped} omitidas')
    return out_json, ann_id


print('Preprocesando Dataset C (filtrado)...')
print('  (usa curva espinal si está disponible)')
aid = 1
TRAIN_C, aid = build_dataset(train_f,    'train', 'c', aid, use_spine_map=True)
VAL_C,   aid = build_dataset(val_f,      'val',   'c', aid, use_spine_map=True)
TEST_C,  aid = build_dataset(test_f,     'test',  'c', aid, use_spine_map=True)

print('\nPreprocesando Dataset D (completo)...')
TRAIN_D, aid = build_dataset(train_full, 'train', 'd', aid, use_spine_map=True)
VAL_D,   aid = build_dataset(val_full,   'val',   'd', aid, use_spine_map=True)
TEST_D,  aid = build_dataset(test_full,  'test',  'd', aid, use_spine_map=True)

print('\n✔ Datasets preprocesados listos')

In [ ]:
# ── Registrar en Detectron2 ───────────────────────────────────
# image_root='/' porque file_name ya tiene ruta absoluta
for name, jp in [
    ('spine_train_c', TRAIN_C), ('spine_val_c', VAL_C), ('spine_test_c', TEST_C),
    ('spine_train_d', TRAIN_D), ('spine_val_d', VAL_D), ('spine_test_d', TEST_D),
]:
    if name in DatasetCatalog:
        DatasetCatalog.remove(name)
        MetadataCatalog.remove(name)
    register_coco_instances(name, {}, str(jp), '/')
    MetadataCatalog.get(name).thing_classes = CLASS_NAMES
    print(f'✔ {name}')

dicts_c = DatasetCatalog.get('spine_train_c')
dicts_d = DatasetCatalog.get('spine_train_d')
print(f'\nC filtrado train: {len(dicts_c)} imgs')
print(f'D completo train: {len(dicts_d)} imgs')

In [ ]:
# ── Verificación visual ───────────────────────────────────────
from detectron2.utils.visualizer import Visualizer
if len(dicts_c) > 0:
    s   = dicts_c[random.randint(0, len(dicts_c)-1)]
    img = cv2.cvtColor(cv2.imread(s['file_name']), cv2.COLOR_BGR2RGB)
    vis = Visualizer(img, metadata=MetadataCatalog.get('spine_train_c'),
                     scale=0.5).draw_dataset_dict(s)
    plt.figure(figsize=(6, 10))
    plt.imshow(vis.get_image())
    plt.title(f'{Path(s["file_name"]).name}\n{len(s["annotations"])} vértebras')
    plt.axis('off'); plt.tight_layout(); plt.show()
    print('✔ Si las máscaras se ven sobre las vértebras → continúa')
else:
    print('⚠ dicts_c vacío — revisar build_dataset')

---
## 5 — Configuración y Entrenamiento

In [ ]:
def build_cfg(output_dir, train_ds, val_ds, num_classes, max_iter=6000):
    """
    max_iter=6000:
    ~101 imgs / batch 2 = 50 iter/época → 6000 iter ≈ 120 épocas
    ~174 imgs / batch 2 = 87 iter/época → 6000 iter ≈ 69 épocas
    """
    cfg = get_cfg()
    cfg.merge_from_file(
        model_zoo.get_config_file(
            'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'
        )
    )
    cfg.DATASETS.TRAIN           = (train_ds,)
    cfg.DATASETS.TEST            = (val_ds,)
    cfg.MODEL.WEIGHTS            = model_zoo.get_checkpoint_url(
        'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'
    )
    cfg.DATALOADER.NUM_WORKERS   = 2
    cfg.SOLVER.IMS_PER_BATCH     = 2
    cfg.SOLVER.BASE_LR           = 0.002
    cfg.SOLVER.MAX_ITER          = max_iter
    cfg.SOLVER.WARMUP_ITERS      = 500
    cfg.SOLVER.WARMUP_FACTOR     = 1.0 / 500
    cfg.SOLVER.STEPS             = (int(max_iter*.70), int(max_iter*.88))
    cfg.SOLVER.GAMMA             = 0.1
    cfg.SOLVER.CHECKPOINT_PERIOD = 1000
    cfg.SOLVER.WEIGHT_DECAY      = 0.0001
    cfg.SOLVER.MOMENTUM          = 0.9
    cfg.SOLVER.AMP.ENABLED       = True
    cfg.TEST.EVAL_PERIOD         = 1000
    cfg.MODEL.ROI_HEADS.NUM_CLASSES          = num_classes
    cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST    = 0.25
    cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST      = 0.50
    cfg.MODEL.ANCHOR_GENERATOR.SIZES         = [[16],[32],[64],[128],[256]]
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.5,1.0,2.0]]
    cfg.MODEL.BACKBONE.FREEZE_AT             = 0
    cfg.INPUT.MIN_SIZE_TRAIN  = (800, 1024)
    cfg.INPUT.MAX_SIZE_TRAIN  = 1333
    cfg.INPUT.MIN_SIZE_TEST   = 1024
    cfg.INPUT.MAX_SIZE_TEST   = 1333
    cfg.INPUT.RANDOM_FLIP     = 'horizontal'
    cfg.OUTPUT_DIR            = str(output_dir)
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    return cfg


class SpineTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, 'inference')
        return COCOEvaluator(dataset_name, output_dir=output_folder)

    @classmethod
    def build_train_loader(cls, cfg):
        augs = [
            T.RandomFlip(prob=0.5, horizontal=True, vertical=False),
            T.RandomRotation(angle=[-15, 15]),
            # Brillo/contraste suave — el preprocesamiento ya normalizó la imagen
            T.RandomBrightness(intensity_min=0.85, intensity_max=1.15),
            T.RandomContrast(intensity_min=0.85,   intensity_max=1.15),
            T.ResizeShortestEdge((800,1024), max_size=1333, sample_style='choice'),
        ]
        return d2_data.build_detection_train_loader(
            cfg, mapper=d2_data.DatasetMapper(cfg, is_train=True, augmentations=augs)
        )


n_train_c = len(train_f)
n_train_d = len(train_full)
iter_c    = max(4000, (n_train_c // 2) * 120)  # ~120 épocas
iter_d    = max(4000, (n_train_d // 2) * 80)   # ~80 épocas

cfg_c = build_cfg(OUTPUT_C, 'spine_train_c', 'spine_val_c', NUM_CLASSES, iter_c)
cfg_d = build_cfg(OUTPUT_D, 'spine_train_d', 'spine_val_d', NUM_CLASSES, iter_d)

print(f'✔ Config C: {iter_c} iter (~120 épocas con {n_train_c} imgs)')
print(f'✔ Config D: {iter_d} iter (~80 épocas con {n_train_d} imgs)')

In [ ]:
print('=' * 60)
print('MODELO C — MaskRCNN + preprocesamiento espinal (filtrado)')
print('=' * 60)
trainer_c = SpineTrainer(cfg_c)
trainer_c.resume_or_load(resume=False)
trainer_c.train()
print(f'✔ Modelo C completado')

print('\n' + '=' * 60)
print('MODELO D — MaskRCNN + preprocesamiento espinal (completo)')
print('=' * 60)
trainer_d = SpineTrainer(cfg_d)
trainer_d.resume_or_load(resume=False)
trainer_d.train()
print(f'✔ Modelo D completado')

---
## 6 — Evaluación

In [ ]:
cfg_c.MODEL.WEIGHTS = str(OUTPUT_C / 'model_final.pth')
cfg_d.MODEL.WEIGHTS = str(OUTPUT_D / 'model_final.pth')
predictor_c = DefaultPredictor(cfg_c)
predictor_d = DefaultPredictor(cfg_d)
print('✔ Modelos C y D cargados')

print('\n=== AP COCO — Modelo C ===')
eval_c = COCOEvaluator('spine_test_c', output_dir=str(OUTPUT_C/'eval'))
print(inference_on_dataset(predictor_c.model,
      build_detection_test_loader(cfg_c,'spine_test_c'), eval_c))

print('\n=== AP COCO — Modelo D ===')
eval_d = COCOEvaluator('spine_test_d', output_dir=str(OUTPUT_D/'eval'))
print(inference_on_dataset(predictor_d.model,
      build_detection_test_loader(cfg_d,'spine_test_d'), eval_d))

In [ ]:
# ── Dice e IoU por clase ──────────────────────────────────────
def gt_from_png(mask_path, h=1024, w=1024):
    m = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
    if m is None: return {c:np.zeros((h,w),np.uint8) for c in range(NUM_CLASSES)}
    if m.ndim==3: m=m[:,:,0]
    if m.shape!=(h,w): m=cv2.resize(m,(w,h),interpolation=cv2.INTER_NEAREST)
    return {c:(m==c+1).astype(np.uint8) for c in range(NUM_CLASSES)}

def dice_iou(p, g):
    p,g = p.astype(bool), g.astype(bool)
    inter=(p&g).sum(); union=(p|g).sum()
    return (2*inter/(p.sum()+g.sum()) if (p.sum()+g.sum())>0 else 1.0,
            inter/union if union>0 else 1.0)

def eval_model(predictor, split_df, model_name):
    dice_cls={c:[] for c in range(NUM_CLASSES)}
    iou_cls ={c:[] for c in range(NUM_CLASSES)}
    l5_log  =[]
    for _, row in split_df.iterrows():
        ip  = DATASET_ROOT / row[COL_IMAGE]
        mp  = DATASET_ROOT / row[COL_MASK]
        bp  = DATASET_ROOT / row[COL_BINARY]
        stem= ip.stem
        curve=load_spine_curve(stem, METRICS_DIR)
        proc =preprocess_image(ip, bp, curve, use_spine_map=True)
        if proc is None: continue
        H,W  = 1024, 1024
        gt   = gt_from_png(mp, H, W)
        out  = predictor(proc)
        inst = out['instances'].to('cpu')
        pred = {c:np.zeros((H,W),np.uint8) for c in range(NUM_CLASSES)}
        if len(inst)>0:
            ca,ma,sa=inst.pred_classes.numpy(),inst.pred_masks.numpy(),inst.scores.numpy()
            for c in range(NUM_CLASSES):
                idx=np.where(ca==c)[0]
                if len(idx)==0: continue
                pred[c]=ma[idx[np.argmax(sa[idx])]].astype(np.uint8)
        for c in range(NUM_CLASSES):
            if gt[c].sum()==0: continue
            d,iou=dice_iou(pred[c],gt[c])
            dice_cls[c].append(d); iou_cls[c].append(iou)
            if c==16:
                l5_log.append({'image':ip.name,'model':model_name,
                               'split':'Scoliosis' if stem.startswith('S_') else 'Normal',
                               'dice':d,'iou':iou,'gt_px':int(gt[c].sum()),
                               'pred_px':int(pred[c].sum()),'detected':pred[c].sum()>0})
    return dice_cls, iou_cls, l5_log


print('Evaluando C y D sobre test filtrado (comparación justa)...')
dice_c, iou_c, l5_c = eval_model(predictor_c, test_f,    'C_MaskRCNN_filt')
dice_d, iou_d, l5_d = eval_model(predictor_d, test_f,    'D_MaskRCNN_full')
print('✔ Listo')

In [ ]:
# ── Tabla comparativa ─────────────────────────────────────────
prev_csv = DRIVE_ROOT / 'models' / 'comparacion_3modelos.csv'
if prev_csv.exists():
    prev      = pd.read_csv(prev_csv)
    dice_a_v  = dict(zip(range(NUM_CLASSES), prev['dice_A'].values))
    dice_b_v  = dict(zip(range(NUM_CLASSES), prev['dice_B'].values))
    dice_bp_v = dict(zip(range(NUM_CLASSES), prev['dice_Bplus'].values))
    print('✔ A/B/B+ cargados desde CSV')
else:
    dice_a_v  = dict(zip(range(17),[0.7044,0.6511,0.4522,0.3042,0.3174,0.2620,
                                     0.2250,0.1543,0.1328,0.1670,0.0589,0.0925,
                                     0.1416,0.2121,0.3132,0.5111,0.2434]))
    dice_b_v  = dict(zip(range(17),[0.7175,0.5860,0.4112,0.3048,0.3467,0.3218,
                                     0.3279,0.2988,0.3254,0.2820,0.3091,0.3380,
                                     0.3580,0.3989,0.4532,0.5111,0.3031]))
    dice_bp_v = dict(zip(range(17),[0.6691,0.5908,0.4279,0.2903,0.2888,0.3075,
                                     0.3033,0.2974,0.2997,0.2979,0.3026,0.3148,
                                     0.3819,0.3502,0.4131,0.5111,0.3126]))
    print('⚠ Usando valores de respaldo')

print(f'\n{"Clase":<6} {"A":>7} {"B":>7} {"B+":>7} {"C-Filt":>8} {"D-Full":>8} {"Mejor":>6}')
print('─'*56)
rows=[]
for c in range(NUM_CLASSES):
    da =dice_a_v.get(c,0.); db =dice_b_v.get(c,0.); dbp=dice_bp_v.get(c,0.)
    dc =np.mean(dice_c[c]) if dice_c[c] else 0.
    dd =np.mean(dice_d[c]) if dice_d[c] else 0.
    bl =['A','B','B+','C','D'][[da,db,dbp,dc,dd].index(max(da,db,dbp,dc,dd))]
    tag=' ◄L5' if c==16 else ''
    print(f'{CLASS_NAMES[c]:<6} {da:>7.4f} {db:>7.4f} {dbp:>7.4f} {dc:>8.4f} {dd:>8.4f} {bl:>6}{tag}')
    rows.append({'clase':CLASS_NAMES[c],'dice_A':da,'dice_B':db,'dice_Bplus':dbp,
                 'dice_C':dc,'dice_D':dd,'mejor':bl,
                 'iou_C':np.mean(iou_c[c]) if iou_c[c] else 0.,
                 'iou_D':np.mean(iou_d[c]) if iou_d[c] else 0.})
print('─'*56)
ma=np.mean([r['dice_A'] for r in rows]); mb=np.mean([r['dice_B'] for r in rows])
mbp=np.mean([r['dice_Bplus'] for r in rows])
mc=np.mean([r['dice_C'] for r in rows]); md=np.mean([r['dice_D'] for r in rows])
print(f'{"MEAN":<6} {ma:>7.4f} {mb:>7.4f} {mbp:>7.4f} {mc:>8.4f} {md:>8.4f}')
print(f'\n  A  YOLOv8 completo         : {ma:.4f}')
print(f'  B  YOLOv8 filtrado         : {mb:.4f}')
print(f'  B+ YOLOv8 filtrado+pos     : {mbp:.4f}')
print(f'  C  MaskRCNN filtrado+preproc: {mc:.4f}  Δ vs B: {mc-mb:+.4f}')
print(f'  D  MaskRCNN completo+preproc: {md:.4f}  Δ vs A: {md-ma:+.4f}')
print(f'  Paper anterior              : 0.7400')
print(f'\n  Impacto filtrado  en YOLOv8  (A→B): {mb-ma:+.4f}')
print(f'  Impacto filtrado  en MaskRCNN(D→C): {mc-md:+.4f}')
print(f'  Impacto MaskRCNN  filtrado vs YOLO (C vs B): {mc-mb:+.4f}')
pd.DataFrame(rows).to_csv(DRIVE_ROOT/'models'/'comparacion_5modelos.csv', index=False)
print('\n✔ Tabla guardada')

In [ ]:
# ── Gráfico comparativo 5 modelos ─────────────────────────────
fig=plt.figure(figsize=(24,14))
gs=gridspec.GridSpec(2,2,figure=fig,hspace=0.45,wspace=0.35)
x=np.arange(NUM_CLASSES); w=0.16

ax1=fig.add_subplot(gs[0,:])
for off,(lbl,col,vals) in enumerate([
    ('A YOLOv8 completo',    '#3498db',[r['dice_A']     for r in rows]),
    ('B YOLOv8 filtrado',    '#2ecc71',[r['dice_B']     for r in rows]),
    ('B+ YOLOv8 filt+pos',   '#e67e22',[r['dice_Bplus'] for r in rows]),
    ('C MaskRCNN filt+prep', '#9b59b6',[r['dice_C']     for r in rows]),
    ('D MaskRCNN full+prep', '#e74c3c',[r['dice_D']     for r in rows]),
]):
    ax1.bar(x+(off-2)*w,vals,w,label=lbl,color=col,alpha=0.85,edgecolor='k')
ax1.axhline(0.70,color='red', ls='--',lw=1.5,label='Umbral 0.70')
ax1.axhline(0.74,color='gray',ls=':' ,lw=1.5,label='Paper 0.74')
ax1.set_xticks(x); ax1.set_xticklabels(CLASS_NAMES,rotation=45)
ax1.set_ylabel('Dice'); ax1.set_ylim(0,1.1)
ax1.set_title('Dice por vértebra — 5 estrategias',fontsize=12)
ax1.legend(fontsize=8,ncol=3); ax1.grid(axis='y',alpha=0.3)

# Región anatómica
ax2=fig.add_subplot(gs[1,0])
groups={'T1-T6':range(0,6),'T7-T12':range(6,12),'L1-L5':range(12,17),'GLOBAL':range(0,17)}
xg=np.arange(len(groups)); wg=0.15
dc_v={c:np.mean(dice_c[c]) if dice_c[c] else 0. for c in range(NUM_CLASSES)}
dd_v={c:np.mean(dice_d[c]) if dice_d[c] else 0. for c in range(NUM_CLASSES)}
for off2,(lbl,col,vd) in enumerate([
    ('A','#3498db',dice_a_v),('B','#2ecc71',dice_b_v),
    ('B+','#e67e22',dice_bp_v),('C','#9b59b6',dc_v),('D','#e74c3c',dd_v)
]):
    yv=[np.mean([vd.get(c,0) for c in rng]) for rng in groups.values()]
    bars=ax2.bar(xg+(off2-2)*wg,yv,wg,label=lbl,color=col,alpha=0.85,edgecolor='k')
    for bar,v in zip(bars,yv):
        ax2.text(bar.get_x()+bar.get_width()/2,v+0.01,f'{v:.2f}',
                 ha='center',fontsize=6,rotation=90)
ax2.axhline(0.70,color='red',ls='--',lw=1.5); ax2.axhline(0.74,color='gray',ls=':',lw=1.5)
ax2.set_xticks(xg); ax2.set_xticklabels(list(groups.keys()))
ax2.set_ylabel('Dice'); ax2.set_ylim(0,1.1)
ax2.set_title('Región anatómica'); ax2.legend(fontsize=8); ax2.grid(axis='y',alpha=0.3)

# Delta C vs D
ax3=fig.add_subplot(gs[1,1])
delta=[rows[c]['dice_C']-rows[c]['dice_D'] for c in range(NUM_CLASSES)]
ax3.bar(x,delta,color=['#9b59b6' if d>=0 else '#e74c3c' for d in delta],edgecolor='k',alpha=0.85)
ax3.axhline(0,color='black',lw=1.5)
ax3.set_xticks(x); ax3.set_xticklabels(CLASS_NAMES,rotation=45)
ax3.set_ylabel('Δ Dice (C_filt − D_full)')
ax3.set_title('Impacto del filtrado en MaskRCNN',fontsize=11)
ax3.grid(axis='y',alpha=0.3)

plt.suptitle('Comparación Final — A|B|B+(YOLOv8) vs C|D (MaskRCNN+Preproc)',
             fontsize=12,fontweight='bold')
plt.savefig(str(DRIVE_ROOT/'models'/'comparacion_5modelos.png'),dpi=150,bbox_inches='tight')
plt.show(); print('✔ Gráfico guardado')

In [ ]:
# ── Análisis L5 ───────────────────────────────────────────────
if l5_c or l5_d:
    l5_cd = pd.DataFrame(l5_c + l5_d)
    print('═══ L5 ═══════════════════════════════════════════════')
    print(l5_cd.groupby('model')[['dice','detected']].agg(
        {'dice':['mean','median','std'],'detected':'mean'}
    ).round(4).to_string())
    print()
    print(l5_cd.groupby(['model','split'])['dice'].agg(
        ['mean','median','count']
    ).round(4).to_string())

    prev_l5 = DRIVE_ROOT / 'models' / 'l5_comparacion.csv'
    l5_all  = pd.concat([pd.read_csv(prev_l5), l5_cd], ignore_index=True) \
              if prev_l5.exists() else l5_cd

    fig, axes = plt.subplots(1,3,figsize=(18,5))
    colors={'A_completo':'#3498db','B_filtrado':'#2ecc71','B+':'#e67e22',
            'C_MaskRCNN_filt':'#9b59b6','D_MaskRCNN_full':'#e74c3c'}
    for mdl in l5_all['model'].unique():
        sub=l5_all[l5_all['model']==mdl]['dice']
        axes[0].hist(sub,bins=10,alpha=0.5,edgecolor='k',
                     color=colors.get(mdl,'gray'),label=f'{mdl} μ={sub.mean():.3f}')
    axes[0].axvline(0.52,color='gray',ls=':',lw=1.5); axes[0].axvline(0.70,color='red',ls='--',lw=1.5)
    axes[0].set_title('L5 todos los modelos'); axes[0].legend(fontsize=7)

    for idx,(mdl,col) in enumerate([('C_MaskRCNN_filt','#9b59b6'),('D_MaskRCNN_full','#e74c3c')]):
        sub_m=l5_cd[l5_cd['model']==mdl]
        bp=axes[idx+1].boxplot(
            [sub_m[sub_m['split']=='Normal']['dice'].values,
             sub_m[sub_m['split']=='Scoliosis']['dice'].values],
            labels=['Normal','Scoliosis'],patch_artist=True)
        for box in bp['boxes']: box.set_facecolor(col); box.set_alpha(0.7)
        axes[idx+1].axhline(0.70,color='red',ls='--',lw=1.5)
        axes[idx+1].set_title(f'{mdl.split("_")[-1]}\nL5 Normal vs Scoliosis')
        axes[idx+1].set_ylabel('Dice L5')

    plt.suptitle('L5 — C (filtrado) vs D (completo) con preprocesamiento espinal')
    plt.tight_layout()
    plt.savefig(str(DRIVE_ROOT/'models'/'l5_5modelos.png'),dpi=150,bbox_inches='tight')
    plt.show()
    l5_all.to_csv(DRIVE_ROOT/'models'/'l5_5modelos.csv',index=False)
    print('✔ Análisis L5 guardado')

In [ ]:
# ── Visualización: Original | GT | C | D ─────────────────────
n_rows=test_f[test_f[COL_IMAGE].str.startswith('Normal')]
s_rows=test_f[test_f[COL_IMAGE].str.startswith('Scoliosis')]
samples=pd.concat([
    n_rows.sample(min(2,len(n_rows)),random_state=SEED),
    s_rows.sample(min(2,len(s_rows)),random_state=SEED)
])

rng_pal=np.random.RandomState(0)
pal=rng_pal.randint(60,230,(NUM_CLASSES,3),dtype=np.uint8)

def make_ov(img_rgb, masks_dict):
    o=img_rgb.copy()
    for c,m in masks_dict.items():
        if m.sum()==0: continue
        col=tuple(int(x) for x in pal[c])
        cl=np.zeros_like(img_rgb); cl[m==1]=col
        o=cv2.addWeighted(o,.65,cl,.35,0)
        cnts,_=cv2.findContours(m,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(o,cnts,-1,col,2)
        M=cv2.moments(m)
        if M['m00']>0:
            cx,cy=int(M['m10']/M['m00']),int(M['m01']/M['m00'])
            cv2.putText(o,CLASS_NAMES[c],(cx-12,cy+5),cv2.FONT_HERSHEY_SIMPLEX,.4,(255,255,255),1)
    return o

for _,row in samples.iterrows():
    ip=DATASET_ROOT/row[COL_IMAGE]; mp=DATASET_ROOT/row[COL_MASK]
    bp=DATASET_ROOT/row[COL_BINARY]; stem=ip.stem
    curve=load_spine_curve(stem,METRICS_DIR)
    proc=preprocess_image(ip,bp,curve,use_spine_map=True)
    proc_rgb=cv2.cvtColor(proc,cv2.COLOR_BGR2RGB)
    orig_rgb=cv2.cvtColor(cv2.imread(str(ip)),cv2.COLOR_BGR2RGB)
    gt=gt_from_png(mp)

    def get_pred(pred_model):
        out=pred_model(proc); inst=out['instances'].to('cpu')
        pred={c:np.zeros((1024,1024),np.uint8) for c in range(NUM_CLASSES)}
        if len(inst)>0:
            ca,ma,sa=inst.pred_classes.numpy(),inst.pred_masks.numpy(),inst.scores.numpy()
            for c in range(NUM_CLASSES):
                idx=np.where(ca==c)[0]
                if len(idx)==0: continue
                pred[c]=ma[idx[np.argmax(sa[idx])]].astype(np.uint8)
        return pred

    tipo='Normal' if 'Normal' in row[COL_IMAGE] else 'Scoliosis'
    fig,axes=plt.subplots(1,4,figsize=(24,8))
    axes[0].imshow(orig_rgb);                     axes[0].set_title('Original')
    axes[1].imshow(make_ov(proc_rgb,gt));          axes[1].set_title('GT (img preproc)')
    axes[2].imshow(make_ov(proc_rgb,get_pred(predictor_c))); axes[2].set_title('C — filtrado+preproc')
    axes[3].imshow(make_ov(proc_rgb,get_pred(predictor_d))); axes[3].set_title('D — completo+preproc')
    for ax in axes: ax.axis('off')
    plt.suptitle(f'{ip.name} ({tipo})',fontsize=11)
    plt.tight_layout(); plt.show()

In [ ]:
# ── Guardar y resumen ─────────────────────────────────────────
save_dir=DRIVE_ROOT/'models'; os.makedirs(save_dir,exist_ok=True)
shutil.copy(OUTPUT_C/'model_final.pth', save_dir/'maskrcnn_C_filt_preproc.pth')
shutil.copy(OUTPUT_D/'model_final.pth', save_dir/'maskrcnn_D_full_preproc.pth')
print('✔ Modelos C y D guardados en Drive')

def rmean(vd,rng): return np.mean([vd.get(c,0) for c in rng])
l5c_m=pd.DataFrame(l5_c)['dice'].mean() if l5_c else 0.
l5d_m=pd.DataFrame(l5_d)['dice'].mean() if l5_d else 0.

print('\n'+'='*70)
print('RESUMEN FINAL — 5 modelos')
print('='*70)
print(f'''
A  YOLOv8m completo (174):             Dice={ma:.4f}
B  YOLOv8m filtrado (101):             Dice={mb:.4f}
B+ YOLOv8m filtrado+posicional:        Dice={mbp:.4f}
C  MaskRCNN filtrado+preprocesamiento: Dice={mc:.4f}  Δ vs B: {mc-mb:+.4f}
D  MaskRCNN completo+preprocesamiento: Dice={md:.4f}  Δ vs A: {md-ma:+.4f}
Paper anterior (YOLOv8m, 134 imgs):    Dice=0.7400

Impacto filtrado  YOLOv8  (A→B): {mb-ma:+.4f}
Impacto filtrado  MaskRCNN(D→C): {mc-md:+.4f}

Dice L5:  C={l5c_m:.4f}  D={l5d_m:.4f}  Paper=0.5200
Lumbares: C={rmean(dc_v,range(12,17)):.4f}  D={rmean(dd_v,range(12,17)):.4f}
''')
print('='*70)